In [1]:
import os
import json 
import shutil
import nltk
from statistics import mean

In [2]:
def jaccard_similarity(output1, output2):
    set1 = set(output1)
    set2 = set(output2)
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    return intersection / union

def calculate_bleu(reference, hypothesis):
    reference = [reference]
    hypothesis = hypothesis
    reference_tokens = [nltk.word_tokenize(ref) for ref in reference]
    hypothesis_tokens = nltk.word_tokenize(hypothesis)
    # Berechne den BLEU-Score
    bleu_score = nltk.translate.bleu_score.sentence_bleu(reference_tokens, hypothesis_tokens)

    return bleu_score

In [3]:
def json_to_text(data):
    if isinstance(data, dict):
        if "AND" in data:
            left = json_to_text(data["AND"]["left"])
            right = json_to_text(data["AND"]["right"])
            return f"({left} AND {right})"
        elif "OR" in data:
            left = json_to_text(data["OR"]["left"])
            right = json_to_text(data["OR"]["right"])
            return f"({left} OR {right})"
        elif "NOT" in data:
            inner = json_to_text(data["NOT"]["left"])
            return f"(NOT {inner})"
        elif "raw_text" in data:
            return data["raw_text"]
    return ""
def read_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

In [4]:
model_name = "Llama-3-70B-Instruct_5_shot"
model_path = f"../model_output/{model_name}/ready"
failed_inner_path = f"../model_output/{model_name}/failed_inner"
label_path = "p2"
base_model_output_path = "../model_output"

In [5]:
model_names = [d for d in os.listdir(base_model_output_path) if os.path.isdir(os.path.join(base_model_output_path, d))]

for model_name in model_names:
    model_path = os.path.join(base_model_output_path, model_name, "ready")
    failed_inner_path = os.path.join(base_model_output_path, model_name, "failed_inner")

    label_files = os.listdir(label_path)
    model_files = os.listdir(model_path)
    bleu_scores = []
    jaccard_similarities = []

    for label_file in label_files:
        base_name = label_file.split("_parsed_2.json")[0]

        # Extract model and nshot (taking into account that there might be two "temp" components)
        parts = model_name.split("_")
        model = "_".join(parts[:-2])
        nshot = parts[-2]
        model_file = f"{model}_{base_name}_{nshot}_shot.json"

        if model_file in model_files:
            label_file_path = os.path.join(label_path, label_file)
            model_file_path = os.path.join(model_path, model_file)

            label_data = read_json(label_file_path)
            model_data = read_json(model_file_path)

            try:
                label_text = json_to_text(label_data)
                model_text = json_to_text(model_data)

                # Berechnung der Metriken
                bleu_score = calculate_bleu(reference=label_text, hypothesis=model_text)
                jaccard_score = jaccard_similarity(label_text, model_text)
                bleu_scores.append(bleu_score)
                jaccard_similarities.append(jaccard_score)

                print(f"Label: {label_file}")
                print(f"Model: {model_file}")
                print(f"BLEU Score: {bleu_score}")
                print(f"Jaccard Similarity: {jaccard_score}")

            except KeyError as e:
                print(f"Error processing file {model_file}: {e}")
                failed_model_path = os.path.join(failed_inner_path, model_file)
                shutil.move(model_file_path, failed_model_path)

    # Durchschnittswerte berechnen
    if bleu_scores:
        average_bleu = mean(bleu_scores)
        average_jaccard = mean(jaccard_similarities)

        print(f"Durchschnittlicher BLEU Score für {model_name}: {average_bleu}")
        print(f"Durchschnittliche Jaccard Ähnlichkeit für {model_name}: {average_jaccard}")

Label: NCT00094861_inc_parsed_2.json
Model: Llama-3-70B-Instruct_NCT00094861_inc_5_shot.json
BLEU Score: 0.6728960751840685
Jaccard Similarity: 0.9672131147540983
Error processing file Llama-3-70B-Instruct_NCT00122070_exc_5_shot.json: 'right'
Label: NCT00122070_inc_parsed_2.json
Model: Llama-3-70B-Instruct_NCT00122070_inc_5_shot.json
BLEU Score: 0.7726951981286995
Jaccard Similarity: 0.9761904761904762
Error processing file Llama-3-70B-Instruct_NCT00182520_exc_5_shot.json: 'right'
Label: NCT00182520_inc_parsed_2.json
Model: Llama-3-70B-Instruct_NCT00182520_inc_5_shot.json
BLEU Score: 0.36732261676418904
Jaccard Similarity: 0.9464285714285714
Label: NCT00183885_exc_parsed_2.json
Model: Llama-3-70B-Instruct_NCT00183885_exc_5_shot.json
BLEU Score: 0.7113351368598995
Jaccard Similarity: 0.8913043478260869
Label: NCT00183885_inc_parsed_2.json
Model: Llama-3-70B-Instruct_NCT00183885_inc_5_shot.json
BLEU Score: 0.6588987082934227
Jaccard Similarity: 0.9423076923076923
Label: NCT00198913_exc_p

C:\Users\e-aut\anaconda3\lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
C:\Users\e-aut\anaconda3\lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
C:\Users\e-aut\anaconda3\lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnin

Label: NCT01116882_exc_parsed_2.json
Model: Llama-3-70B-Instruct_NCT01116882_exc_5_shot.json
BLEU Score: 0.5884144491308606
Jaccard Similarity: 0.9552238805970149
Label: NCT01116882_inc_parsed_2.json
Model: Llama-3-70B-Instruct_NCT01116882_inc_5_shot.json
BLEU Score: 0.8112982857531542
Jaccard Similarity: 1.0
Error processing file Llama-3-70B-Instruct_NCT01116973_exc_5_shot.json: 'right'
Error processing file Llama-3-70B-Instruct_NCT01116973_inc_5_shot.json: 'right'
Label: NCT01117181_inc_parsed_2.json
Model: Llama-3-70B-Instruct_NCT01117181_inc_5_shot.json
BLEU Score: 0.8136534196802713
Jaccard Similarity: 0.9811320754716981
Label: NCT01118871_exc_parsed_2.json
Model: Llama-3-70B-Instruct_NCT01118871_exc_5_shot.json
BLEU Score: 0.7913001060808317
Jaccard Similarity: 0.9411764705882353
Label: NCT01118871_inc_parsed_2.json
Model: Llama-3-70B-Instruct_NCT01118871_inc_5_shot.json
BLEU Score: 0.6192496572799826
Jaccard Similarity: 1.0
Label: NCT01175044_exc_parsed_2.json
Model: Llama-3-70B

In [32]:
# Durchschnittswerte berechnen
average_bleu = mean(bleu_scores)
average_jaccard = mean(jaccard_similarities)

print(f"Durchschnittlicher BLEU Score: {average_bleu}")
print(f"Durchschnittliche Jaccard Ähnlichkeit: {average_jaccard}")

Durchschnittlicher BLEU Score: 0.6386162177097935
Durchschnittliche Jaccard Ähnlichkeit: 0.9362588664018855


In [30]:
# Label
label_text = json_to_text(label_data)
print(label_text)

(((Provide written informed consent before beginning any study related activities AND Be between age 18 and 55 years) AND Be able to speak, read and write English and follow simple instructions for completing self-rated scales) AND (Meet DSM-IV criteria for BPD AND as assessed by the Structured Clinical Interview for DSM-IV Personality Disorders (SCID-II).))


In [31]:
# Model
model_text = json_to_text(model_data)
print(model_text)

((Provide written informed consent before beginning any study related activities AND (Be between age 18 and 55 years AND (Be able to speak, read and write English AND and follow simple instructions for completing self-rated scales))) AND Meet DSM-IV criteria for BPD as assessed by the Structured Clinical Interview for DSM-IV Personality Disorders (SCID-II))


In [33]:
calculate_bleu(reference=label_text , hypothesis=model_text)

0.7726951981286995

In [36]:
jaccard_similarity(label_text,model_text)

0.9761904761904762